## Deep Weighted Monte Carlo
This notebook contains an implementation of the Deep Weight Monte Carlo approach of [Kunsagi-Mate et al. 2022](https://arxiv.org/abs/2208.14038).

In [ ]:
import QuantLib as ql
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.nn.functional as F
import math
import numpy as np
from numpy.linalg import cholesky, norm
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from joblib import dump
import os

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and prepare the Synthetic Implied Vol Data

In [ ]:
base_filename = 'bergomi_option_grid'

In [ ]:
bergomi_data_list = list()
for i in range(0,20):
    filename = base_filename + str(i) + '.csv'
    if os.path.exists(filename):
        new_bergomi_data = pd.read_csv(filename)
        new_bergomi_data['i'] += i * 250
        bergomi_data_list.append(new_bergomi_data) 
    
bergomi_data = pd.concat(bergomi_data_list)
bergomi_data.reset_index(drop=True, inplace=True)

In [ ]:
bergomi_data

In [ ]:
drop_list = ['eta', 'rho', 'H', 'xi_0', 'time']
bergomi_data.drop(drop_list, axis=1, inplace=True)

In [ ]:
pd.set_option('display.max_rows', 10)

In [ ]:
bergomi_data[bergomi_data['i']==0]

In [ ]:
strikes = sorted(bergomi_data['K'].unique())
maturities = sorted(bergomi_data['tau'].unique())

In [ ]:
def plotdfcolumn(filename, dfin, cmap, colname, display=True):
    # Pivot the DataFrame (FIX: use keyword args)
    vol_surface = dfin.pivot(index='K', columns='tau', values=colname)

    # Optional: sort axes for cleaner surface
    vol_surface = vol_surface.sort_index().sort_index(axis=1)

    # Prepare grid
    X, Y = np.meshgrid(vol_surface.columns.values,
                       vol_surface.index.values)
    Z = vol_surface.values

    # Create figure
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Plot surface
    surf = ax.plot_surface(X, Y, Z,
                           cmap=cmap,
                           edgecolor='none',
                           antialiased=False)

    # Labels
    ax.set_xlabel('Maturity T')
    ax.set_ylabel('Strike K')
    ax.set_zlabel(colname)

    # Optional: invert x-axis cleanly instead of reversed limits
    ax.invert_xaxis()

    # View angle
    ax.view_init(azim=30)

    # Colorbar
    fig.colorbar(surf, shrink=0.5, aspect=5)

    # Save
    plt.savefig(filename, dpi=300, bbox_inches='tight')

    if not display:
        plt.close(fig)

In [ ]:
file_exiv = 'ExampleVols.png'

In [ ]:
plotdfcolumn(file_exiv, bergomi_data[bergomi_data['i'] == 10], 'viridis', 'iv')

In [ ]:
file_exivbw = 'ExampleVols_bw.png'

In [ ]:
plotdfcolumn(file_exivbw, bergomi_data[bergomi_data['i'] == 10], 'gray', 'iv')

In [ ]:
vol_surface = bergomi_data[bergomi_data['i'] == 0].pivot(
    index='K',
    columns='tau',
    values='iv'
)

In [ ]:
vol_surface

In [ ]:
def df_to_3d_tensor(df):
    # Get unique values of 'i', 'K', and 'mat'
    unique_i = sorted(df['i'].unique())
    unique_K = sorted(df['K'].unique())
    unique_mat = sorted(df['tau'].unique())
    
    # Initialize an empty list to store 2D tensors
    tensor_list = []
    
    # Loop through each unique value of 'i'
    for i_val in unique_i:
        # Filter the DataFrame for the current value of 'i'
        df_i = df[df['i'] == i_val]
        
        # Pivot the DataFrame
        pivot_df = df_i.pivot(index='K', columns='tau', values='iv')
        
        # Reindex the pivot table to ensure all K and mat values are represented
        pivot_df = pivot_df.reindex(index=unique_K, columns=unique_mat, fill_value=0)
        
        # Convert the pivot table to a 2D tensor and append to the list
        tensor_list.append(torch.tensor(pivot_df.values, dtype=torch.float32))
    
    # Stack the list of 2D tensors into a 3D tensor
    tensor_3d = torch.stack(tensor_list)
    
    return tensor_3d

In [ ]:
minmax_scalar = MinMaxScaler()
bergomi_data['iv'] = minmax_scalar.fit_transform(bergomi_data[['iv']])

In [ ]:
bergomi_data.head()

In [ ]:
opttensor = df_to_3d_tensor(bergomi_data).to(device)
tensor_2d = opttensor.view(opttensor.size(0), -1)

In [ ]:
opttensor.shape

In [ ]:
nan_mask = torch.isnan(tensor_2d).any(dim=1)
tensor_cleaned = tensor_2d[~nan_mask]

In [ ]:
print(tensor_cleaned.shape)

In [ ]:
no_samples = len(tensor_cleaned)
train_len = int(0.8 * no_samples)

In [ ]:
train_t, test_t = tensor_cleaned[:train_len], tensor_cleaned[train_len:]

In [ ]:
def create_dataloaders(train_tensor, test_tensor, batch_size=32):
    train_dataset = TensorDataset(train_tensor)
    test_dataset = TensorDataset(test_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader

In [ ]:
train_t

In [ ]:
batch_size = 100

In [ ]:
train_loader, test_loader = create_dataloaders(train_t, test_t, batch_size)

### Build the VAE model

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = F.silu(self.fc1(x))
        h = F.silu(self.fc2(h))
        mean = self.fc_mean(h)
        log_var = self.fc_log_var(h)
        return mean, log_var

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h = F.silu(self.fc1(x))
        h = F.silu(self.fc2(h))
        out = torch.sigmoid(self.fc4(h))
        return out

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mean, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mean + eps * std

    def forward(self, x):
        mean, log_var = self.encoder(x)
        z = self.reparameterize(mean, log_var)
        x_recon = self.decoder(z)
        return x_recon, mean, log_var

In [ ]:
def vae_loss(recon_x, x, mean, log_var, beta):
    recon_loss = F.mse_loss(recon_x, x)
    kld_loss = torch.mean(-0.5 * torch.sum(1 + log_var - mean ** 2 - log_var.exp(), dim = 1), dim = 0)
    return recon_loss + kld_loss * beta, recon_loss, kld_loss

In [ ]:
model = VAE(88, 16, 4).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Train the encoder-decoder pair

In [ ]:
num_epochs = 1000
beta = 0.00005

In [ ]:
loss_lst = list()
reloss_list = list()
kl_loss_list = list()
tqdm_epoch = trange(num_epochs)

for epoch in tqdm_epoch:
    sum_loss = 0.0
    num_items = 0
    re_loss = 0.0
    kl_loss = 0.0
    for [data] in train_loader:
        cur_batch_size = len(data)
        optimizer.zero_grad()
        recon_batch, mean, log_var = model(data)
        loss, rc_loss, kl_loss = vae_loss(recon_batch, data, mean, log_var, beta)
        loss.backward()
        optimizer.step()
        
        sum_loss += loss.item() * cur_batch_size
        re_loss += rc_loss.item() * cur_batch_size
        kl_loss += kl_loss.item() * cur_batch_size
        num_items += cur_batch_size
    
    ave_loss = sum_loss / num_items
    ave_re_loss = re_loss / num_items
    ave_kl_loss = kl_loss / num_items
    loss_lst.append(ave_loss)
    tqdm_epoch.set_description('Ave Loss: {:5e}, Ave Rec Loss: {:5e},'
                               'Ave KL Loss: {:5e}'.format(ave_loss, ave_re_loss, ave_kl_loss))

In [ ]:
epochs = range(0, num_epochs)
plt.figure(figsize=(12,6)) 
        
plt.plot(epochs, loss_lst, color='k', linestyle='-')

plt.xlabel('Epochs')
plt.ylabel('Average Loss')
plt.yscale('log')
plt.savefig('VSVAELoss.png', dpi=300, bbox_inches='tight')

Calculate reconstruction loss on train and test datasets

In [ ]:
def reconstruction_loss(model, dataloader):
    num_items = 0
    re_loss = 0.0
    model.eval()
    for [data] in dataloader:
        cur_batch_size = len(data)
        recon_batch, _, _ = model(data)
        _, rc_loss, _ = vae_loss(recon_batch, data, mean, log_var, beta)
        
        re_loss += rc_loss.item() * cur_batch_size
        num_items += cur_batch_size
    
    return re_loss / num_items

In [ ]:
train_rl = reconstruction_loss(model, train_loader)
test_rl = reconstruction_loss(model, test_loader)

In [ ]:
train_rl

In [ ]:
test_rl

In [ ]:
test_t

In [ ]:
rec_test_t = model(test_t)

In [ ]:
rec_test_t

In [ ]:
train_t

In [ ]:
rec_train_t = model(train_t)

In [ ]:
rec_train_t

In [ ]:
dump(minmax_scalar, 'bergomi_scaler.bin', compress=True)

In [ ]:
model_path = "vaebergomi.pth"
torch.save(model.state_dict(), model_path)

### Impact of different latent space elements

In [ ]:
def latent_z_impact():
    z = torch.zeros((4 * 5, 4)).to(device)
    sample_points = [-2.0, -1.0, 0, 1.0, 2.0]
    for j in range((len(sample_points))):
        for i in range(4):
            z[i*5 + j, i] = sample_points[j]
    return z

In [ ]:
latent_z = latent_z_impact()

In [ ]:
latent_z

In [ ]:
vols = model.decoder(latent_z).reshape((20, 11, 8)).detach().cpu()

In [ ]:
vols.shape

In [ ]:
strikes

In [ ]:
def generate_3d_plots(data_tensor, strikes, maturities, latent_z, colormap='viridis', save_filename='plots.png'):
    num_examples = data_tensor.size(0)
    num_rows = 4
    num_cols = 5

    def float_row_to_string(row):
        return "[" + ", ".join([str(elem) for elem in row]) + "]"
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(25, 20), subplot_kw={'projection': '3d'})

    for i in range(num_examples):
        string_representation = float_row_to_string(latent_z[i])
        row = i // num_cols
        col = i % num_cols

        ax = axes[row, col]

        strike_mesh, maturity_mesh = torch.meshgrid(torch.arange(len(strikes)), torch.arange(len(maturities)))
        strike_mesh, maturity_mesh = strike_mesh.float(), maturity_mesh.float()

        ax.plot_surface(strike_mesh, maturity_mesh, data_tensor[i], cmap=colormap)
        ax.set_xlabel('Strike')
        ax.set_ylabel('Maturity')
        ax.set_zlabel('Impled Vol')
        ax.set_title(string_representation)

        ax.set_xticks(torch.arange(0, len(strikes), step=2))
        ax.set_xticklabels([strikes[i] for i in range(0, len(strikes), 2)], rotation=45)
        ax.set_yticks(torch.arange(len(maturities)))
        ax.set_yticklabels(maturities)

    plt.tight_layout()
    plt.savefig(save_filename)

In [ ]:
clatent_z = latent_z.detach().cpu().numpy()

In [ ]:
generate_3d_plots(vols, strikes, maturities, clatent_z, colormap='plasma', save_filename='LatentShifts.png')

In [ ]:
generate_3d_plots(vols, strikes, maturities, clatent_z, colormap='gray', save_filename='LatentShifts_BW.png')

### Deep Weighted Monte Carlo

In [ ]:
class DeepWMCNetwork(nn.Module):
    def __init__(self, latent_size, output_size):
        super(DeepWMCNetwork, self).__init__()
        self.fc1 = nn.Linear(latent_size, 32)
        self.fc2 = nn.Linear(32, 64)
        self.fc3 = nn.Linear(64, output_size)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return self.softmax(out)

In [ ]:
def simulate_asset_paths(S0, r, sigma, T, dt, num_steps, num_paths, device='cpu'):
    # Initialize the tensor to store paths on the specified device
    paths = [torch.zeros((num_paths, 1), dtype=torch.float32, device=device) for i in range(num_steps+1)]
    
    # Set the initial asset price
    paths[0][:,0] = S0
    
    # Precompute constants
    drift = (r - 0.5 * sigma ** 2) * dt
    sqrdt = torch.sqrt(torch.tensor(dt, device=device))
    diffusion = sigma * sqrdt
    
    # Generate paths
    for t in range(1, num_steps + 1):
        Z = torch.randn(num_paths, device=device)
        paths[t][:,0] = paths[t-1][:, 0] * torch.exp(drift + diffusion * Z)
    return torch.cat(paths, axis=1)

In [ ]:
def price_calls_puts(paths, weights, strikes, maturities, r, device):
    nmat = np.array(maturities)
    dfs = np.exp(-r * nmat)
    tdfs = torch.tensor(dfs, dtype=torch.float32, device=device)
    tdfs = tdfs.unsqueeze(0).unsqueeze(0)
    T = maturities[-1]
    batch_size = weights.shape[0]
    
    num_steps = paths.shape[1]
    idxs = [int(num_steps * mt / T) - 1 for mt in maturities]
       
    S = paths[:, idxs]  # shape: (num_paths, len(maturities))
    
    # Expand strikes and maturities to match the shape of S for broadcasting
    strikes_expanded = torch.tensor(strikes, dtype=torch.float32, device=device).view(-1, 1)
    maturities_expanded = torch.tensor(maturities, dtype=torch.float32, device=device).view(1, -1)
    
    # Compute call and put payoffs for each maturity and strike
    tcpayoff = torch.maximum(S.unsqueeze(1) - strikes_expanded, torch.tensor(0.0, device=device))
    tppayoff = torch.maximum(strikes_expanded - S.unsqueeze(1), torch.tensor(0.0, device=device))
    dcpayoffs = tcpayoff * tdfs
    dppayoffs = tppayoff * tdfs
    
    calls = torch.tensordot(weights, dcpayoffs, dims=([1], [0]))
    puts = torch.tensordot(weights, dppayoffs, dims=([1], [0]))
    return calls, puts

In [ ]:
S = 1 # Initial stock price
r = 0.0   # Risk-free rate
sigma = 0.25  # Volatility
n = 80     # Number of time steps
T = maturities[-1]
dt = T / n
m = 4096 # Number of paths

In [ ]:
paths = simulate_asset_paths(S, r, sigma, maturities[-1],  dt, n, m, device)

In [ ]:
paths.shape

In [ ]:
weights = torch.ones((16, m), dtype = torch.float32, device=device) / m

In [ ]:
calls, puts = price_calls_puts(paths, weights, strikes, maturities, r, device)

In [ ]:
calls

In [ ]:
z = torch.randn((16, 4), device=device)
ivs = model.decoder(z)

In [ ]:
ivs.shape

In [ ]:
class DeepWMCModel(nn.Module):
    def __init__(self, S0, r, prior_sigma, dt, num_steps, num_paths,
                 maturities, strikes, latent_size, device='cpu'):
        super(DeepWMCModel, self).__init__()
        self.S0 = S0
        self.r = r
        self.prior_sigma = prior_sigma
        self.dt = dt
        self.num_steps = num_steps
        self.num_paths = num_paths
        self.T = max(maturities)
        self.maturities = maturities
        self.strikes = strikes
        self.deepwmcnet = DeepWMCNetwork(latent_size, num_paths)
        # Generate the prior paths
        self.paths = simulate_asset_paths(S0, r, prior_sigma, maturities[-1], 
                                          dt, num_steps, num_paths, device=device)
        
    def forward(self, z):
        weights = self.deepwmcnet(z)
        return price_calls_puts(self.paths, weights, self.strikes, self.maturities, r, device)

In [ ]:
def calculate_option_prices(sigma, strikes, maturities, spot_price, r, device, is_call=True):
    
    # Ensure strikes and maturities are tensors on the same device as the input tensor
    strikes = torch.tensor(strikes, dtype=torch.float32, device=device)
    maturities = torch.tensor(maturities, dtype=torch.float32, device=device)
    
    # Create grid for strikes and maturities
    K = strikes.view(1, -1, 1)  # shape: 1 x strikes x 1
    T = maturities.view(1, 1, -1)  # shape: 1 x 1 x maturities
    
    # Black-Scholes calculations
    S = torch.tensor(spot_price, dtype=torch.float32, device=device)
    r = torch.tensor(r, dtype=torch.float32, device=device)

    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * torch.sqrt(T))
    d2 = d1 - sigma * torch.sqrt(T)
    
    if is_call:
        option_prices = S * torch.distributions.Normal(0, 1).cdf(d1) - \
            K * torch.exp(-r * T) * torch.distributions.Normal(0, 1).cdf(d2)
    else:
        option_prices = K * torch.exp(-r * T) * torch.distributions.Normal(0, 1).cdf(-d2) - \
            S * torch.distributions.Normal(0, 1).cdf(-d1)
    
    return option_prices

In [ ]:
def train_dwc(vae_decoder, dwc_model, n_iter, optimizer, latent_size, batches, 
              mini_batches, minscale, maxscale, device=device):
    vae_decoder.eval()
    tqdm_epoch = trange(n_iter)
    mmdiff = maxscale - minscale
    no_mats = len(dwc_model.maturities)
    no_strikes = len(dwc_model.strikes)
    
    history = {
        'loss': list(),
        'calls': list(),
        'vcalls': list(),
        'vols': list()
    }
    z = torch.randn((batches, latent_size), device=device)
    
    for it in tqdm_epoch:
        imb = 0
        loss_list = list()
        for _ in range(batches // mini_batches):
            
            dwc_model.train()
            optimizer.zero_grad()
        
            # Forward pass
            iz = z[imb:imb+mini_batches,:]
            ivs = vae_decoder(iz)
            calls, puts = dwc_model(iz)
        
            # unscale implied vols
            sivs = ivs * mmdiff + minscale
            rsivs = sivs.reshape((64, no_strikes, no_mats))
            vcalls = calculate_option_prices(rsivs, dwc_model.strikes, dwc_model.maturities, 
                                         dwc_model.S0, dwc_model.r, device, True)
            vputs = calculate_option_prices(rsivs, dwc_model.strikes, dwc_model.maturities, 
                                         dwc_model.S0, dwc_model.r, device, False)
        
            loss = F.mse_loss(vcalls, calls) + F.mse_loss(vputs, puts)
            loss.backward()
        
            optimizer.step()
            loss_list.append(loss.item())
        
            # Update progress bar
            tqdm_epoch.set_description(f"Iteration {it+1}/{n_iter} \
                - Loss: {loss:.8e}")
            imb += mini_batches
        
        ave_loss = sum(loss_list) / len(loss_list)
        print(f"Interation {it+1} / {n_iter} average loss: {ave_loss:.8e} ")
        # Store metrics
        history['loss'].append(ave_loss)
        history['calls'].append(calls.detach().cpu().numpy())
        history['vcalls'].append(vcalls.detach().cpu().numpy())
        history['vols'].append(rsivs.detach().cpu().numpy())
    return history

In [ ]:
dwc_model = DeepWMCModel(S, r, sigma, dt, n, m, maturities, strikes, 4, device=device).to(device)
dwc_opt = optim.Adam(dwc_model.parameters(), lr=0.001)

In [ ]:
n_iter = 1000
batch_size = 4096
mini_batches = 64
minscale = minmax_scalar.data_min_.item()
maxscale = minmax_scalar.data_max_.item()

In [ ]:
history = train_dwc(model.decoder, dwc_model, n_iter, dwc_opt, 4, batch_size, 
                    mini_batches, minscale, maxscale, device=device)

In [ ]:
iterations = range(0, n_iter)
plt.figure(figsize=(12,6)) 
        
plt.plot(iterations, history['loss'], color='k', linestyle='-')

plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.yscale('log')
plt.savefig('DWCLoss.png', dpi=300, bbox_inches='tight')

In [ ]:
dwc_model.eval()

In [ ]:
z = torch.randn((1, 4), device=device)

In [ ]:
ivs = model.decoder(z)
calls, puts = dwc_model(z)
        
# unscale implied vols
mmdiff = maxscale - minscale
no_mats = len(dwc_model.maturities)
no_strikes = len(dwc_model.strikes)
sivs = ivs * mmdiff + minscale
rsivs = sivs.reshape((1, no_strikes, no_mats))
vcalls = calculate_option_prices(rsivs, dwc_model.strikes, dwc_model.maturities, 
                                dwc_model.S0, dwc_model.r, device, True)
vputs = calculate_option_prices(rsivs, dwc_model.strikes, dwc_model.maturities, 
                                dwc_model.S0, dwc_model.r, device, False)

In [ ]:
z

In [ ]:
rsivs

In [ ]:
colormap = 'viridis'
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': '3d'})
strike_mesh, maturity_mesh = torch.meshgrid(torch.arange(len(strikes)), torch.arange(len(maturities)))
strike_mesh, maturity_mesh = strike_mesh.float(), maturity_mesh.float()

ax.plot_surface(strike_mesh, maturity_mesh, rsivs.squeeze(0).detach().cpu(), cmap=colormap)
ax.set_xlabel('Strike')
ax.set_ylabel('Maturity')
ax.set_zlabel('Implied Vol')

ax.set_xticks(torch.arange(0, len(strikes), step=2))
ax.set_xticklabels([strikes[i] for i in range(0, len(strikes), 2)], rotation=45)
ax.set_yticks(torch.arange(len(maturities)))
ax.set_yticklabels(maturities)
plt.savefig("vaegenvol.png")

In [ ]:
colormap = 'gray'
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': '3d'})
strike_mesh, maturity_mesh = torch.meshgrid(torch.arange(len(strikes)), torch.arange(len(maturities)))
strike_mesh, maturity_mesh = strike_mesh.float(), maturity_mesh.float()

ax.plot_surface(strike_mesh, maturity_mesh, rsivs.squeeze(0).detach().cpu(), cmap=colormap)
ax.set_xlabel('Strike')
ax.set_ylabel('Maturity')
ax.set_zlabel('Implied Vol')

ax.set_xticks(torch.arange(0, len(strikes), step=2))
ax.set_xticklabels([strikes[i] for i in range(0, len(strikes), 2)], rotation=45)
ax.set_yticks(torch.arange(len(maturities)))
ax.set_yticklabels(maturities)
plt.savefig("vaegenvol_bw.png")

In [ ]:
cgrid = calls.squeeze(0).detach().cpu()

In [ ]:
vcgrid = vcalls.squeeze(0).detach().cpu()

In [ ]:
pgrid = puts.squeeze(0).detach().cpu()

In [ ]:
vpgrid = vputs.squeeze(0).detach().cpu()

In [ ]:
def subplot_prices(grid, strikes, maturities, title, row, col, colormap, azim=0):
    
    ax = axes[row, col]

    strike_mesh, maturity_mesh = torch.meshgrid(torch.arange(len(strikes)), torch.arange(len(maturities)))
    strike_mesh, maturity_mesh = strike_mesh.float(), maturity_mesh.float()

    ax.view_init(azim=azim)
    ax.plot_surface(strike_mesh, maturity_mesh, grid, cmap=colormap)
    ax.set_xlabel('Strike')
    ax.set_ylabel('Maturity')
    ax.set_zlabel('Prices')
    ax.set_title(title)

    ax.set_xticks(torch.arange(0, len(strikes), step=2))
    ax.set_xticklabels([strikes[i] for i in range(0, len(strikes), 2)], rotation=45)
    ax.set_yticks(torch.arange(len(maturities)))
    ax.set_yticklabels(maturities)

In [ ]:
colormap = 'viridis'
fig, axes = plt.subplots(2, 2, figsize=(25, 20), subplot_kw={'projection': '3d'})

subplot_prices(cgrid, strikes, maturities, "Simulated Calls", 0, 0, colormap, 45)
subplot_prices(vcgrid, strikes, maturities, "VAE+BS Calls", 0, 1, colormap, 45)
subplot_prices(pgrid, strikes, maturities, "Simulated Puts", 1, 0, colormap, 135)
subplot_prices(vpgrid, strikes, maturities, "VAE+BS Puts", 1, 1, colormap, 135)

plt.tight_layout()
plt.savefig("callputprices_dwc.png")

In [ ]:
colormap = 'gray'
fig, axes = plt.subplots(2, 2, figsize=(25, 20), subplot_kw={'projection': '3d'})

subplot_prices(cgrid, strikes, maturities, "Simulated Calls", 0, 0, colormap, 45)
subplot_prices(vcgrid, strikes, maturities, "VAE+BS Calls", 0, 1, colormap, 45)
subplot_prices(pgrid, strikes, maturities, "Simulated Puts", 1, 0, colormap, 135)
subplot_prices(vpgrid, strikes, maturities, "VAE+BS Puts", 1, 1, colormap, 135)

plt.tight_layout()
plt.savefig("callputprices_dwc_bw.png")

In [ ]:
dwcmodel_path = "dwcmodel.pth"
torch.save(dwc_model.state_dict(), dwcmodel_path)